# Contingency-monitored line automation

.aux file used in Sensitivities > Multiple Elements > Select Interfaces > shift factor tab in PowerWorld. The goal for this notebook is to automatically spit out an .aux file given a list of all the branches.

The file will outut 2 * (N choose 2) = n(n-1) constraints, where N is the number of branches.

Each constraint will have a monitored line (the line that is being monitored for overload) and a contingency (the line that is being taken out of service (OPEN)). We produce 1 constraint for each possible combination of monitored line and contingency.

File is formatted as
```json
...preamble...
{
    1 "<constraint name 1>" <MW> 0 "" "FROM -> TO" "NO " 0.0 0.0 0.0 "" "" "" "Yes" <MW> 0.0
    2 "<constraint name 2>" <MW> 0 "" "FROM -> TO" "NO " 0.0 0.0 0.0 "" "" "" "Yes" <MW> 0.0
    ...
}

...preamble...
{
"<constraint name 1>" "BRANCH <from> <to> <circuit (always 1)>" "NO " 1.000000 // monitored line
"<constraint name 1>" "BRANCHOPEN <from> <to> <circuit (always 1)>" "" "" // contingency
"<constraint name 2>" "BRANCH <from> <to> <circuit (always 1)>" "NO " 1.000000 // monitored line
"<constraint name 2>" "BRANCHOPEN <from> <to> <circuit (always 1)>" "" "" // contingency
}
```

#### Update 21 Feb: We can actually set MW to 0 for all without causing issue since PowerWorld will recalculate the shift factors based on the monitored line and contingency.

In [27]:
import pandas as pd

In [28]:
# INPUT FILES HERE
model = "300-bus"
bus_file = f"../raw-data/larger-model-processed/{model}/buses.csv"
branch_file = f"../raw-data/larger-model-processed/{model}/branches.csv"

In [29]:
branches = pd.read_csv(branch_file, header=1)
branches.index.name = "Branch ID"
branches.index = branches.index + 1
branches.head(2)

,From Number,From Name,To Number,To Name,Circuit,Status,Branch Device Type,Xfrmr,R,X,B,Lim MVA A,Lim MVA B,Lim MVA C
Branch ID,,,,,,,,,,,,,,
1,1,1,5,1,1,Closed,Line,NO,0.001,0.006,0.0,0.0,0.0,0.0
2,2,1,6,1,1,Closed,Line,NO,0.001,0.009,0.0,0.0,0.0,0.0


In [30]:
branches.loc[1]

From Number                1
From Name                  1
To Number                  5
To Name                    1
Circuit                    1
Status                Closed
Branch Device Type      Line
Xfrmr                     NO
R                      0.001
X                      0.006
B                        0.0
Lim MVA A                0.0
Lim MVA B                0.0
Lim MVA C                0.0
Name: 1, dtype: object

In [31]:
# Create all pairings
constraints = [
    # Stored as (monitored_line(id, from, to), contingency_line(id, from, to))
]

branch_ids = branches.index.tolist()
for monitored_line in branch_ids:
    for contingency_line in branch_ids:
        if monitored_line == contingency_line:
            continue
        constraints.append(
            [
                ( # Monitored Line
                    monitored_line,
                    branches.loc[monitored_line]["From Number"],
                    branches.loc[monitored_line]["To Number"]
                ),
                ( # Contingency Line
                    contingency_line,
                    branches.loc[contingency_line]["From Number"],
                    branches.loc[contingency_line]["To Number"]
                )
            ]
        )
        
        

In [32]:
# Create aux file

constraints_list = []
interface_elements_list = []
i = 1
for (monitored_line, from_1, to_1), (contingency, from_2, to_2) in constraints:
   constraint_name = f"MONITOR_{monitored_line}_{from_1}-{to_1}_CONTINGENCY_{contingency}_{from_2}-{to_2}"
   constraint_postamble = '0.0 0.0 "" "FROM -> TO" "NO " 0.0 0.0 0.0 "" "" "" "Yes" 0.0 0.0'
   constraints_list.append(f'       {i} {constraint_name} {constraint_postamble}')
   
   # Monitored line
   interface_monitored_line = f"BRANCH {from_1} {to_1} 1"
   interface_elements_list.append(f'       {constraint_name} "{interface_monitored_line}" "NO " 1.000000')
   
   # Contingency line
   interface_contingency_line = f"BRANCHOPEN {from_2} {to_2} 1"
   interface_elements_list.append(f'       {constraint_name} "{interface_contingency_line}" "" ""')
   
   i += 1

constraints_str = '\n'.join(constraints_list)
interface_elements_str = '\n'.join(interface_elements_list)

FILE_TEMPLATE = f"""Interface (Number,Name,MW,LimitUsed,Percent,MonDirection,MonBoth,LimitMWA,LimitMWB,LimitMWC,
   LimitMWANeg,LimitMWBNeg,LimitMWCNeg,HasCTG,MWCTG,MWBase)
{{
{constraints_str}
}}

InterfaceElement (InterfaceName,Element,MeterFar,Weight)
{{
{interface_elements_str}
}}
"""

with open(f"../raw-data/larger-model-processed/{model}/aux_file.aux", "w", encoding='utf-8', newline="") as f:
    f.write(FILE_TEMPLATE)